In [2]:
import joblib 
from transformers import AutoTokenizer, AutoModelForSequenceClassification 
import torch

/Users/mandeepsingh/workspace/Projects/movies-reviews-sentiment-analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_model(model_type, model_name):
    if model_type == 'ml':
        model_path = f"../models/ml/{model_name}.joblib"
        model = joblib.load(model_path)
        return model, None 
    
    elif model_type == 'transformer':
        model_path = f"../models/transformers/{model_name}"

        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(model_path)

        device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
        model.to(device)
        model.eval()

        return model, tokenizer
    
    else:
        raise ValueError("model type must be 'ml' or 'transformer'")

In [4]:
LABEL_MAP = {
    0: 'Negative',
    1: 'Positive'
}

In [5]:
def predict(text, model_type, model_name, LABEL_MAP):
    model, tokenizer = load_model(model_type, model_name)

    if model_type == "ml":
        pred = model.predict([text])[0]
        return LABEL_MAP[pred]
        
    elif model_type == "transformer":
        device = next(model.parameters()).device

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=256
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)
            pred = torch.argmax(probs, dim=-1).item()

        return LABEL_MAP[pred]

In [6]:
text = "Movie is great"
model_type = "ml"
model_name = "logistic_regression"
predict(text, model_type, model_name, LABEL_MAP)

'Positive'

In [7]:
text = "Movie is great"
model_type = "transformer"
model_name = "v0"
predict(text, model_type, model_name, LABEL_MAP)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2543.91it/s, Materializing param=pre_classifier.weight]                                  


'Positive'